# VAHDAM — Post-Purchase Hero Creatives Generator

Batch-generates all **21 email hero images** (PP1–PP22, PP7 excluded) with the new  
**VAHDAM Ashwagandha Coffee+** packet swapped into each scene.

### How to use
1. Run **Cell 1** — installs packages  
2. Run **Cell 2** — paste your API key  
3. Run **Cell 3** — upload your new packet PNG  
4. Run **Cell 4** — test with 2 images first  
5. Run **Cell 5** — full batch (all 21)  
6. Run **Cell 6** — download a zip of all outputs

Supports **OpenAI** (`gpt-image-1`) and **Gemini** (`gemini-2.5-flash-image`).

In [ ]:
# ── Cell 1 · Install dependencies ─────────────────────────────────────────
import subprocess, sys

BACKEND = "openai"   # ← change to "gemini" if you prefer

if BACKEND == "openai":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "openai", "pillow", "requests"])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "google-genai", "pillow", "requests"])

print(f"✅ packages ready for {BACKEND} backend")

In [ ]:
# ── Cell 2 · API key ───────────────────────────────────────────────────────
import os, getpass

if BACKEND == "openai":
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")
    print("✅ OpenAI key set")
else:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
    print("✅ Gemini key set")

In [ ]:
# ── Cell 3 · Upload packet image ───────────────────────────────────────────
# In Colab: a file picker will appear. Select your new Coffee+ packet PNG.
# Locally:  just set PACKET_PATH to the file path manually.

PACKET_PATH = None

try:
    from google.colab import files as colab_files
    print("Running in Colab — upload your packet image below:")
    uploaded = colab_files.upload()
    if uploaded:
        PACKET_PATH = list(uploaded.keys())[0]
        print(f"✅ uploaded: {PACKET_PATH}")
    else:
        raise ValueError("No file uploaded")
except ImportError:
    # Running locally — set the path here
    PACKET_PATH = "./new-packet.png"   # ← change to your local path
    if not os.path.exists(PACKET_PATH):
        raise FileNotFoundError(f"Packet image not found at: {PACKET_PATH}")
    print(f"✅ packet image: {PACKET_PATH}")

# Preview
from PIL import Image
import IPython.display as display
img = Image.open(PACKET_PATH)
img.thumbnail((400, 400))
display.display(img)
print(f"   size: {Image.open(PACKET_PATH).size}")

In [ ]:
# ── Cell 4 · Load prompts + define generator ───────────────────────────────
import json, base64, time, pathlib, requests as req
from pathlib import Path

# Fetch prompts.json straight from GitHub (always up to date)
PROMPTS_URL = (
    "https://raw.githubusercontent.com/anchittandon-vahdam/"
    "vahdam-trustpilot-mailers/claude/charming-noether-DPvq2/"
    "creatives-pipeline/prompts.json"
)
r = req.get(PROMPTS_URL, timeout=20)
r.raise_for_status()
DATA = r.json()
print(f"✅ loaded {len(DATA['creatives'])} creative prompts from GitHub")

OUT_DIR = Path("./creatives")
OUT_DIR.mkdir(exist_ok=True)

BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "image/avif,image/webp,image/png,image/*,*/*;q=0.8",
    "Referer": "https://www.vahdam.co.uk/",
}

def build_prompt(item):
    parts = [
        "TASK: Produce a premium marketing email hero image.",
        f"PRODUCT (use the attached packet image as the exact reference): {DATA['product_reference']}",
        f"SCENE: {item['prompt']}",
        f"STYLE: {DATA['global_style']}",
    ]
    if item.get("role") == "gift_hero_with_pouch_prop":
        parts.append(
            f"IMPORTANT: Hero subject is the gift ({item.get('gift_item','the gift')}). "
            f"Coffee+ pouch is a supporting prop only."
        )
    else:
        parts.append("IMPORTANT: The Coffee+ pouch is the HERO subject.")
    parts.append("Do NOT show old/previous coffee packaging. Only the new dark forest-green Coffee+ pouch.")
    return "\n".join(parts)


def try_download_source(url):
    """Download original hero for composition reference. Returns bytes or None."""
    if not url:
        return None
    try:
        r = req.get(url, headers=BROWSER_HEADERS, timeout=20)
        if r.status_code == 200 and len(r.content) > 2000:
            return r.content
    except Exception:
        pass
    return None


def generate_openai(item, use_source=True, sleep=2):
    from openai import OpenAI
    client = OpenAI()
    prompt = build_prompt(item)
    images = [open(PACKET_PATH, "rb")]
    src_bytes = try_download_source(item.get("source_url")) if use_source else None
    src_tmp = None
    if src_bytes:
        src_tmp = OUT_DIR / f"_src_{item['id']}.png"
        src_tmp.write_bytes(src_bytes)
        images.append(open(src_tmp, "rb"))
        print(f"  + source reference downloaded")
    try:
        resp = client.images.edit(
            model="gpt-image-1",
            image=images,
            prompt=prompt,
            size="1536x1024",
        )
        dest = OUT_DIR / f"{item['id']}.png"
        dest.write_bytes(base64.b64decode(resp.data[0].b64_json))
        print(f"  ✅ saved {dest}")
        return dest
    except Exception as e:
        print(f"  ❌ ERROR: {e}")
        return None
    finally:
        for fh in images:
            try: fh.close()
            except: pass
        if src_tmp and src_tmp.exists():
            src_tmp.unlink()
        time.sleep(sleep)


def generate_gemini(item, use_source=True, sleep=3):
    from google import genai
    from google.genai import types
    client = genai.Client()
    prompt = build_prompt(item)
    packet_bytes = Path(PACKET_PATH).read_bytes()
    contents = [prompt, types.Part.from_bytes(data=packet_bytes, mime_type="image/png")]
    src_bytes = try_download_source(item.get("source_url")) if use_source else None
    if src_bytes:
        contents.append(types.Part.from_bytes(data=src_bytes, mime_type="image/png"))
        print(f"  + source reference downloaded")
    try:
        resp = client.models.generate_content(model="gemini-2.5-flash-image", contents=contents)
        for part in resp.candidates[0].content.parts:
            if getattr(part, "inline_data", None):
                dest = OUT_DIR / f"{item['id']}.png"
                dest.write_bytes(part.inline_data.data)
                print(f"  ✅ saved {dest}")
                return dest
        print(f"  ⚠️ no image in response")
    except Exception as e:
        print(f"  ❌ ERROR: {e}")
    finally:
        time.sleep(sleep)
    return None


def generate(item, use_source=True):
    print(f"\n[{item['id']}] {item['alt'][:60]}...")
    if BACKEND == "openai":
        return generate_openai(item, use_source=use_source)
    else:
        return generate_gemini(item, use_source=use_source)

print("✅ generator ready")

In [ ]:
# ── Cell 5 · TEST — generate PP1 + PP2 only ───────────────────────────────
# Run this first to validate your key + packet image before the full batch.

import IPython.display as display
from PIL import Image

TEST_IDS = ["PP1", "PP2"]
test_items = [c for c in DATA["creatives"] if c["id"] in TEST_IDS]

for item in test_items:
    path = generate(item, use_source=True)
    if path:
        img = Image.open(path)
        img.thumbnail((700, 400))
        display.display(img)

print("\n--- test complete ---")

In [ ]:
# ── Cell 6 · FULL BATCH — all 21 creatives ────────────────────────────────
# Only run after the test above looks good.

results = {}
for item in DATA["creatives"]:
    path = generate(item, use_source=True)
    results[item["id"]] = str(path) if path else "FAILED"

print("\n=== BATCH COMPLETE ===")
ok  = [k for k,v in results.items() if v != "FAILED"]
err = [k for k,v in results.items() if v == "FAILED"]
print(f"✅ {len(ok)} generated: {ok}")
if err:
    print(f"❌ {len(err)} failed:    {err}")

In [ ]:
# ── Cell 7 · Preview all generated images ─────────────────────────────────
import IPython.display as display
from PIL import Image

for png in sorted(OUT_DIR.glob("PP*.png")):
    print(f"\n{png.name}")
    img = Image.open(png)
    img.thumbnail((700, 400))
    display.display(img)

In [ ]:
# ── Cell 8 · Download everything as a zip ─────────────────────────────────
import zipfile, io

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    for png in sorted(OUT_DIR.glob("PP*.png")):
        zf.write(png, png.name)

buf.seek(0)

try:
    # Colab
    from google.colab import files as colab_files
    with open("vahdam_creatives.zip", "wb") as f:
        f.write(buf.read())
    colab_files.download("vahdam_creatives.zip")
    print("✅ download triggered")
except ImportError:
    # Local — just tell them where the files are
    pngs = sorted(OUT_DIR.glob("PP*.png"))
    print(f"✅ {len(pngs)} images in {OUT_DIR.resolve()}")
    for p in pngs:
        print(f"  {p.name}")